# 06 — Linear Regression
Inspect and evaluate the existing fitted StandardScaler → LinearRegression pipeline. Reuse frozen partitions and bundle, without refitting or altering preprocessing. All performance results concern synthetic observations only.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ml/data/phase3_dataset_manifest.json').is_file())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
approved = Path(os.environ['LOCALAPPDATA']) / 'SmartParking/ml-runtime/Scripts/python.exe'
assert Path(sys.executable).resolve() == approved.resolve()
import numpy as np
import pandas as pd
from ml.features import FEATURE_COLUMNS
from ml.phase4_models import load_predictor, metrics, predictions_agree
from ml.train_phase4 import read_partition, verify_manifest
manifest = json.loads(Path('ml/data/phase3_dataset_manifest.json').read_text(encoding='utf-8'))
results = json.loads(Path('ml/artifacts/phase4/model_results.json').read_text(encoding='utf-8'))
verify_manifest(manifest)
bundle = load_predictor(results['artifact']['path'], results['artifact']['sha256'])
model = bundle.models['linear_regression']
print('Interpreter:', sys.executable)

Interpreter: C:\Users\vighn\AppData\Local\SmartParking\ml-runtime\Scripts\python.exe


## Training mathematics and saved preprocessing
The training objective is min(β,b) Σ(y−(zβ+b))², where z=(x−training_mean)/training_scale. The frozen scaler and coefficients were learned from training rows only. There is no validation/test fit. Correlated predictors give a rank-deficient design; coefficients should not be interpreted causally.
The regressor is unconstrained. Common evaluation and inference use clip(ŷ,0,12), preserve fractional predictions, and report raw metrics plus the number of clipped rows.

In [2]:
x_train, _, train_identity = read_partition(manifest, 'train')
assert train_identity == results['partitions']['train']
scaler = model.named_steps['scale']
assert int(scaler.n_samples_seen_) == len(x_train)
np.testing.assert_allclose(scaler.mean_, x_train.mean(axis=0), rtol=0, atol=1e-10)
display(pd.DataFrame({'feature': FEATURE_COLUMNS, 'coefficient_on_scaled_feature': model.named_steps['regressor'].coef_}).sort_values('coefficient_on_scaled_feature', key=abs, ascending=False).head(10))
print('Training-only scaler rows:', scaler.n_samples_seen_, '| rank:', model.named_steps['regressor'].rank_)
del x_train

,feature,coefficient_on_scaled_feature
32,occupied_mean_60,-0.568707
12,total_occupied,0.381360
23,occupied_lag_1,0.380471
25,occupied_lag_15,0.364940
31,occupied_mean_15,0.271588
10,P11,0.264024
1,P2,0.262809
3,P4,0.261707
2,P3,0.257443
0,P1,0.256777


Training-only scaler rows: 49360.0 | rank: 34


In [3]:
rows = []
for split in ('validation', 'test'):
    x, y, identity = read_partition(manifest, split)
    assert identity == results['partitions'][split]
    raw_prediction = model.predict(x)
    measured = metrics(y, raw_prediction)
    frozen = results[split]['linear_regression']
    assert measured['rows'] == frozen['rows']
    assert measured['clipped_count'] == frozen['clipped_count']
    for kind in ('raw', 'clipped'):
        for key in ('mae', 'rmse', 'r2'):
            np.testing.assert_allclose(measured[kind][key], frozen[kind][key], rtol=0, atol=1e-12)
    named = pd.DataFrame(x, columns=FEATURE_COLUMNS, copy=False)
    bounded = bundle.predict(named, 'linear_regression')
    assert predictions_agree(bounded, np.clip(raw_prediction, 0, 12))
    assert np.isfinite(bounded).all() and ((bounded >= 0) & (bounded <= 12)).all()
    rows.append({'split': split, 'rows': len(y), **measured['clipped'], 'raw_mae': measured['raw']['mae'], 'clipped_count': measured['clipped_count'], 'origin_sha256': identity['origin_sha256']})
    if split == 'test':
        display(pd.DataFrame({'actual_t_plus_15': y[:8], 'raw_prediction': raw_prediction[:8], 'clipped_prediction': bounded[:8]}))
display(pd.DataFrame(rows))
print('PASS: identical rows, raw/clipped metric agreement and saved pipeline prediction interface')

,actual_t_plus_15,raw_prediction,clipped_prediction
0,3.0,3.108356,3.108356
1,3.0,3.117317,3.117317
2,3.0,3.116426,3.116426
3,3.0,3.107633,3.107633
4,3.0,3.104326,3.104326
5,3.0,3.100568,3.100568
6,3.0,3.096379,3.096379
7,3.0,3.091774,3.091774


,split,rows,mae,rmse,r2,raw_mae,clipped_count,origin_sha256
0,validation,10577,0.414362,0.656363,0.954385,0.423693,1302,6c28dca1155df5f02c545ae508f03e006079183464b664...
1,test,10578,0.373729,0.613500,0.950942,0.384557,1590,79f277c74ba6bd39467ba2bf0f72e0312f6795028c6d6a...


PASS: identical rows, raw/clipped metric agreement and saved pipeline prediction interface


## Interpretation
Compare MAE and RMSE separately: a model can reduce squared error yet lose the predeclared MAE selection criterion. Clipping enforces a physical range but does not calibrate forecasts or guarantee generalization. No new model choice is made in this notebook. Original training/test evidence and the Phase 4 PDF remain frozen.